> **Edición pública anonimizada.** Nombres y rutas comerciales se sustituyeron por etiquetas genéricas; no se incluyen bases ni modelos originales. Los resultados históricos se consultan en la tesis adaptada. La demostración pública usa datos y modelos sintéticos.


In [ ]:
from pathlib import Path
import os, sys

# Find the repository independently of the computer or clone location.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / ".tesis-portafolio").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Abre Jupyter desde la carpeta de este repositorio.")
os.chdir(REPO_ROOT / '06 Otros')


# Análisis de colinealidad del modelo predictivo final

Modelo: **regresión logística L2, `C = 0.55`**, entrenada sobre el **Universo H** (22 656 leads) con 24 variables de entrada (§21.4 del notebook final).

Objetivo de este cuaderno: cuantificar la colinealidad entre las variables de entrada del modelo final y
explicar su origen y su efecto. La colinealidad **no sesga las predicciones ni degrada el ordenamiento**
de un modelo regularizado usado para priorizar; sí infla la varianza de los coeficientes individuales,
razón por la cual el capítulo de resultados advierte que los coeficientes no deben leerse de forma aislada.

Contenido:
1. Reconstrucción de la matriz de entrada del modelo
2. Dependencias por construcción (variables derivadas y codificaciones redundantes)
3. Matriz de correlación de las variables numéricas
4. Factor de inflación de la varianza (VIF)
5. Agrupamiento jerárquico de variables por correlación
6. Número de condición de la matriz de diseño
7. Conclusiones

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150, 'font.size': 10})
sns.set_style('whitegrid')
SALIDA = Path('figuras_colinealidad'); SALIDA.mkdir(exist_ok=True)
DATA = Path(f'{REPO_ROOT}/01 Limpieza Código/data/features/leads_enriched_qualified_final.csv')

## 1. Reconstrucción de la matriz de entrada del modelo

Se reproduce exactamente la ingeniería de variables de §21.4: cinco requisitos de completitud definen el
Universo H y se construyen las 24 variables predictoras (10 numéricas y 14 categóricas).

In [ ]:
df = pd.read_csv(DATA, low_memory=False)

CANAL_FAMILIA = {
 'Facebook':'pauta_digital','Instagram':'pauta_digital','Fanpage Líderes':'pauta_digital','Inbox':'pauta_digital',
 'portal_demo':'portal','Página Web':'portal','WhatsApp':'directo','Llamada':'directo',
 'Referencia':'referido','Friends/Family':'referido','Shareholders Sale':'referido',
 'Sede':'presencial','ExpoCasa':'presencial','ExpoCasa Virtual':'presencial','Visita Inédita':'presencial',
 'Open House':'presencial','OpenSale':'presencial','Ferias':'presencial','Evento':'presencial',
 'Actividades MKT':'presencial','Actividad Posicionamiento':'presencial','Lanzamiento':'presencial',
 'Vallas':'exterior','Corporación':'otro','usa':'otro','Prensa':'otro','Radio/tv':'otro'}
INCOME_MID = {'Menos De Q10k':8,'Q 10k A Q13.9k':12,'Q 14k A Q17.9k':16,'Q18k A Q21.9k':20,'Q22k A Q25.9k':24,
 'Q26k A Q29.9k':28,'Q30k A 33.9k':32,'Q34k A 37.9k':36,'Q38k A 41.9k':40,'Q42k A 45.9k':44,'46k A 49.9k':48,'Q50k A Más':55}

d = df.copy()
d['edad_n'] = pd.to_numeric(d['edad'], errors='coerce')
d['ingreso_mid'] = d['ingreso_familiar'].map(INCOME_MID)
d['es_inversion'] = (d['razon_de_compra'] == 'Inversión').astype(int)
d['canal_familia'] = d['canal_de_anuencia'].map(CANAL_FAMILIA).fillna('otro')
d['es_referido'] = (d['canal_familia'] == 'referido').astype(int)
d['edad_x_inversion'] = d['edad_n'] * d['es_inversion']
d['edad_x_ingreso'] = d['edad_n'] * d['ingreso_mid'] / 100.0
d['edad_x_referido'] = d['edad_n'] * d['es_referido']
d['mayor_45'] = (d['edad_n'] >= 45).astype(int)
d['ingreso_x_inversion'] = d['ingreso_mid'] * d['es_inversion']
d['ingreso_alto'] = (d['ingreso_mid'] >= 32).astype(int)

REQ_H = ['fuente_pt2','canal_de_anuencia','canal_de_contacto','ingreso_familiar','edad']
FEATS = ['edad_n','sexo','estado_civil','ingreso_mid','ingreso_familiar','razon_de_compra','canal_de_anuencia',
 'canal_de_contacto','canal_familia','tipo_unidad','opciones_visitadas','participantes_en_credito','lugar_trabajo',
 'lugar_vivienda','campania_tipo','campania_audiencia','es_inversion','es_referido','edad_x_inversion',
 'ingreso_x_inversion','edad_x_ingreso','edad_x_referido','mayor_45','ingreso_alto']
NUM = ['edad_n','ingreso_mid','es_inversion','es_referido','edad_x_inversion','ingreso_x_inversion',
 'edad_x_ingreso','edad_x_referido','mayor_45','ingreso_alto']
CAT = [c for c in FEATS if c not in NUM]

dH = d.loc[d[REQ_H].notna().all(axis=1)].reset_index(drop=True)
print(f'Universo H: {len(dH):,} leads | {len(NUM)} numéricas | {len(CAT)} categóricas')

## 2. Dependencias por construcción

Antes de medir correlaciones conviene identificar las relaciones **exactas**: variables que son función
determinista de otras. Estas son la causa principal de la colinealidad del modelo y provienen de decisiones
de ingeniería de variables, no de los datos.

In [ ]:
dep = pd.DataFrame([
 ('es_referido',        'canal_de_anuencia / canal_familia', 'Indicador: 1 si canal_familia == "referido" (= canal_de_anuencia en {Referencia, Friends/Family, Shareholders Sale}).'),
 ('es_inversion',       'razon_de_compra',                   'Indicador: 1 si razon_de_compra == "Inversión".'),
 ('ingreso_mid',        'ingreso_familiar',                  'Punto medio numérico del rango declarado de ingreso: recodificación 1 a 1 de la misma variable categórica.'),
 ('mayor_45',           'edad_n',                            'Indicador: 1 si edad_n >= 45.'),
 ('ingreso_alto',       'ingreso_mid',                       'Indicador: 1 si ingreso_mid >= 32 (Q30 mil).'),
 ('edad_x_ingreso',     'edad_n, ingreso_mid',               'Producto (dividido entre 100).'),
 ('edad_x_inversion',   'edad_n, es_inversion',              'Producto.'),
 ('edad_x_referido',    'edad_n, es_referido',               'Producto.'),
 ('ingreso_x_inversion','ingreso_mid, es_inversion',         'Producto.'),
], columns=['variable derivada','deriva de','definición'])

print('Verificación de las igualdades exactas:')
print('  es_referido  == (canal_familia == "referido"):',
      bool((dH['es_referido'] == (dH['canal_familia'] == 'referido').astype(int)).all()))
print('  es_inversion == (razon_de_compra == "Inversión"):',
      bool((dH['es_inversion'] == (dH['razon_de_compra'] == 'Inversión').astype(int)).all()))
print('  mayor_45     == (edad_n >= 45):',
      bool((dH['mayor_45'] == (dH['edad_n'] >= 45).astype(int)).all()))
dep

**Lectura.** El modelo recibe, para una misma señal, hasta tres representaciones simultáneas:

- el **canal de anuencia** como variable categórica completa, **`canal_familia`** como su agrupación, y **`es_referido`** como indicador binario de una de sus categorías;
- la **razón de compra** como categórica y **`es_inversion`** como indicador binario de una de sus categorías;
- el **ingreso** como categórica (`ingreso_familiar`), como numérico (`ingreso_mid`) y como indicador (`ingreso_alto`);
- la **edad** como numérico (`edad_n`) y como indicador (`mayor_45`), más cuatro términos de interacción que son productos de las anteriores.

Por eso los coeficientes individuales no son interpretables: la contribución de "ser referido" se reparte entre
`es_referido`, la categoría `canal_de_anuencia = Referencia` y `canal_familia = referido`.

## 3. Matriz de correlación de las variables numéricas

Correlación de Pearson entre las 10 variables numéricas, tras imputación por mediana y estandarización
(la transformación que aplica el `pipeline` del modelo).

In [ ]:
Xn = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(dH[NUM]), columns=NUM)
Xz = pd.DataFrame(StandardScaler().fit_transform(Xn), columns=NUM)
corr = Xz.corr()

fig, ax = plt.subplots(figsize=(9, 7.5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=.5, cbar_kws={'label': 'Correlación de Pearson'}, ax=ax)
ax.set_title('Correlación entre las variables numéricas del modelo final (Universo H)')
plt.tight_layout(); plt.savefig(SALIDA / 'col_correlacion.png', bbox_inches='tight'); plt.show()

pares = [(NUM[i], NUM[j], corr.iloc[i, j]) for i in range(len(NUM)) for j in range(i+1, len(NUM))]
altos = sorted([p for p in pares if abs(p[2]) > 0.7], key=lambda p: -abs(p[2]))
print('Pares con |r| > 0.7:')
for a, b, r in altos:
    print(f'  {a:20s} · {b:20s}  r = {r:+.3f}')

## 4. Factor de inflación de la varianza (VIF)

El VIF de una variable es $1/(1-R^2)$, donde $R^2$ resulta de regresarla contra las demás variables numéricas.
Valores por encima de 5 indican colinealidad moderada y por encima de 10, colinealidad alta.

In [ ]:
vif = {}
for c in NUM:
    otras = [x for x in NUM if x != c]
    r2 = LinearRegression().fit(Xz[otras], Xz[c]).score(Xz[otras], Xz[c])
    vif[c] = np.inf if r2 >= 1 else 1 / (1 - r2)
vif_s = pd.Series(vif).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colores = ['#c44e52' if v > 10 else '#dd8452' if v > 5 else '#55a868' for v in vif_s]
ax.barh(vif_s.index[::-1], vif_s.values[::-1], color=colores[::-1])
ax.axvline(5, color='#888', ls='--', lw=1); ax.axvline(10, color='#c44e52', ls='--', lw=1)
ax.text(5, -0.6, 'VIF = 5', color='#888', fontsize=8); ax.text(10, -0.6, 'VIF = 10', color='#c44e52', fontsize=8)
for i, v in enumerate(vif_s.values[::-1]):
    ax.text(v + 0.3, i, f'{v:.1f}', va='center', fontsize=9)
ax.set(xlabel='VIF', title='Factor de inflación de la varianza — variables numéricas del modelo final')
plt.tight_layout(); plt.savefig(SALIDA / 'col_vif.png', bbox_inches='tight'); plt.show()
vif_s.round(2)

Los VIF altos corresponden, sin excepción, a los **términos de interacción y a los indicadores derivados**:
cada producto (`edad_x_ingreso`, `edad_x_inversion`, …) está fuertemente correlacionado con sus factores,
y cada indicador (`mayor_45`, `ingreso_alto`) con la variable continua de la que se obtiene. Las variables
de origen (`edad_n`, `ingreso_mid`) heredan VIF elevado por participar en esos productos.

## 5. Agrupamiento jerárquico por correlación

Se agrupan las variables numéricas por similitud, usando $1-|r|$ como distancia. Las ramas que se unen
a baja altura son casi intercambiables para el modelo.

In [ ]:
dist = 1 - corr.abs()
Z = linkage(squareform(dist.values, checks=False), method='average')
fig, ax = plt.subplots(figsize=(10, 5))
dendrogram(Z, labels=NUM, ax=ax, color_threshold=0.4, leaf_rotation=35)
ax.axhline(0.3, color='#c44e52', ls='--', lw=1, label='distancia 0,3  (|r| $\\approx$ 0,7)')
ax.set(ylabel='distancia  (1 $-$ |r|)', title='Agrupamiento jerárquico de las variables numéricas por correlación')
ax.legend(); plt.tight_layout(); plt.savefig(SALIDA / 'col_dendrograma.png', bbox_inches='tight'); plt.show()

## 6. Número de condición de la matriz de diseño

El número de condición (cociente entre el mayor y el menor valor singular) mide cuán cerca está la matriz
de ser singular. Por encima de 30 se considera colinealidad problemática para estimación por mínimos
cuadrados ordinarios; la regularización L2 del modelo lo compensa.

In [ ]:
prep = ColumnTransformer([
  ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='_missing_')),
                    ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=30))]), CAT),
  ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), NUM)])
Xfull = prep.fit_transform(dH[FEATS])

sv_num = np.linalg.svd(Xz.values, compute_uv=False)
sv_full = np.linalg.svd(Xfull - Xfull.mean(0), compute_uv=False)
cond_num = sv_num[0] / sv_num[-1]
cond_full = sv_full[0] / sv_full[sv_full > 1e-10][-1]
print(f'Matriz completa (con one-hot): {Xfull.shape[0]:,} x {Xfull.shape[1]}')
print(f'Número de condición — solo bloque numérico estandarizado : {cond_num:6.1f}')
print(f'Número de condición — matriz completa centrada            : {cond_full:6.1f}')
print(f'Valores singulares casi nulos (< 1e-8) en la matriz completa: {int((sv_full < 1e-8).sum())}')

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(range(1, len(sv_num)+1), sv_num, 'o-', color='#4c72b0')
ax.set(xlabel='Componente', ylabel='Valor singular', yscale='log',
       title=f'Espectro de valores singulares del bloque numérico (número de condición = {cond_num:.1f})')
plt.tight_layout(); plt.savefig(SALIDA / 'col_valores_singulares.png', bbox_inches='tight'); plt.show()

El **bloque numérico** tiene número de condición $\approx$ 14: colinealidad presente pero moderada. La **matriz
completa** (con las columnas one-hot) es numéricamente singular: la codificación sin descarte de categoría de
referencia, más la categoría `_missing_` y `es_referido`/`es_inversion` que duplican una categoría, generan
combinaciones lineales exactas entre columnas. Una regresión logística sin regularización no tendría solución
única en esta matriz; la penalización L2 (`C = 0.55`) impone una solución estable repartiendo el peso entre las
columnas dependientes.

## 7. Conclusiones

1. **La colinealidad del modelo es alta y de origen conocido.** Proviene de la ingeniería de variables:
   términos de interacción (productos de otras variables), indicadores de umbral (`mayor_45`, `ingreso_alto`)
   y codificaciones redundantes (`es_referido`, `es_inversion`, `ingreso_mid` duplican información de una
   variable categórica). No proviene de una relación espuria entre los datos.

2. **No afecta el uso previsto del modelo.** El modelo se emplea para **ordenar** leads por probabilidad
   estimada, no para estimar el efecto de cada variable. La colinealidad infla la varianza de los coeficientes
   individuales, pero no sesga las probabilidades predichas ni degrada el PR-AUC ni la estabilidad del
   ordenamiento (la segunda semilla de validación cambió las métricas en menos del 1 %).

3. **Sí impide la lectura de coeficientes uno por uno.** El signo y la magnitud de un coeficiente dependen
   del resto de términos del modelo; por eso la categoría `canal_de_anuencia = Referencia` aparece con un
   coeficiente negativo grande mientras que `es_referido` es la variable más importante por permutación:
   son la misma señal repartida entre dos columnas. La interpretación debe apoyarse en la importancia por
   permutación (que evalúa variables como bloque) y en el análisis SHAP, no en los coeficientes crudos.

4. **Para una versión futura más interpretable** convendría conservar una sola representación por señal
   (la categórica o su indicador, no ambas), centrar las variables antes de formar las interacciones y
   descartar los indicadores de umbral redundantes. Esto reduciría los VIF sin cambiar de forma apreciable
   el desempeño de ordenamiento.

Figuras generadas en `figuras_colinealidad/`: `col_correlacion.png`, `col_vif.png`, `col_dendrograma.png`,
`col_valores_singulares.png`.